In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import os
import seaborn as sns
import sys
import glob
import tqdm
plt.style.use('default')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 22, 'legend.facecolor': 'white', 'legend.framealpha': 1, "legend.frameon": 1, "lines.linewidth": 2})

In [ ]:
from Bio import SeqIO
ref_path = "/extdata4/baeklab/Hyeonseo/m6A/res/ref/isoform/hg38_rna_nrnm.fasta"
fasta_dict = SeqIO.to_dict(SeqIO.parse(ref_path, "fasta"))
fasta_dict = {x.split(".")[0]: y for x, y in fasta_dict.items()}


In [ ]:
sys.path.append("/extdata4/baeklab/Hyeonseo/m6A/modformer")
from utils.utils import parse_refflat
refflat_df = parse_refflat()

In [ ]:
print(refflat_df)

In [ ]:
refflat_df["refseq"] = refflat_df.index
refflat_df.reset_index(inplace=True, drop=True)
refflat_df = refflat_df[refflat_df["refseq"].str.startswith("NM")].copy()
print(refflat_df)

In [ ]:
def get_utr_orf_len(exonstarts, exonends, cdsstart, cdsend, strand):
    
    if exonstarts[0] == cdsstart:
        utr5_len = 0
    else: 
        utr5_end_idx = np.argmax(np.array(exonends) > cdsstart)
        utr5_exonstarts = exonstarts[:utr5_end_idx+1].copy()
        utr5_exonends = exonends[:utr5_end_idx+1].copy()
        utr5_exonends[-1] = cdsstart
        utr5_len = sum(np.array(utr5_exonends) - np.array(utr5_exonstarts))
        
        if utr5_len == 0:
            print(exonstarts, exonends, cdsstart, cdsend, strand)
            raise ValueError("UTR5 length is 0")

    if exonends[-1] == cdsend:
        utr3_len = 0
    else: 
        utr3_start_idx = np.argmax(np.array(exonends) >= cdsend)
        utr3_exonstarts = exonstarts[utr3_start_idx:].copy()
        utr3_exonends = exonends[utr3_start_idx:].copy()
        utr3_exonstarts[0] = cdsend
        utr3_len = sum(np.array(utr3_exonends) - np.array(utr3_exonstarts))

        if utr3_len == 0:
            print(exonstarts, exonends, cdsstart, cdsend, strand)
            raise ValueError("UTR5 length is 0")
        
    if strand == "-":
        utr5_len, utr3_len = utr3_len, utr5_len
    
    tlen = sum(np.array(exonends) - np.array(exonstarts))
    orf_len = tlen - utr5_len - utr3_len
    return utr5_len, orf_len, utr3_len
        

In [ ]:

get_utr_orf_len([439977, 445835], [443494, 445940], 443163, 445940, "-")

In [ ]:

refflat_df[["utr5_len", "orf_len", "utr3_len"]] = refflat_df.apply(lambda x: get_utr_orf_len(x["exonStarts"], x["exonEnds"], x["cdsStart"], x["cdsEnd"], x["strand"]), axis=1, result_type="expand")


In [ ]:
print(refflat_df)

In [ ]:
refflat_df["tlen"] = refflat_df["refseq"].map(lambda x: len(fasta_dict[x]))

print(refflat_df)

In [ ]:
m6a_df["refseq"] = m6a_df["label_id"].str.split(":").str[0]
m6a_df = m6a_df.merge(refflat_df[['refseq', 'utr5_len', 'orf_len', 'utr3_len']], on="refseq", how="inner")

In [ ]:
hek_m6a_df["refseq"] = hek_m6a_df["label_id"].str.split(":").str[0]
hek_m6a_df = hek_m6a_df.merge(refflat_df[['refseq', 'utr5_len', 'orf_len', 'utr3_len']], on="refseq", how="inner")

hela_m6a_df["refseq"] = hela_m6a_df["label_id"].str.split(":").str[0]
hela_m6a_df = hela_m6a_df.merge(refflat_df[['refseq', 'utr5_len', 'orf_len', 'utr3_len']], on="refseq", how="inner")

In [ ]:
utr_df = refflat_df[refflat_df[['utr5_len', 'orf_len', 'utr3_len']].min(axis=1) > 0].copy()

In [ ]:
print(utr_df)

In [ ]:
mean_utr5 = utr_df["utr5_len"].mean()
mean_orf = utr_df["orf_len"].mean()
mean_utr3 = utr_df["utr3_len"].mean()

print(mean_utr5, mean_orf, mean_utr3)

In [ ]:
mean_sum = mean_utr5 + mean_orf + mean_utr3
mean_utr5 /= mean_sum
mean_orf /= mean_sum
mean_utr3 /= mean_sum

print(mean_utr5, mean_orf, mean_utr3)

In [ ]:
def calculate_metagene(utr5_len, orf_len, utr3_len, pos, length_preset = (0.1,0.6,0.3)):
    try:
        bins = [0, utr5_len, utr5_len + orf_len, utr5_len + orf_len + utr3_len]
        length_index = np.digitize(pos, bins) -1
        return (pos - bins[length_index]) / [utr5_len, orf_len, utr3_len][length_index] * length_preset[length_index] + sum(length_preset[:length_index])
    except:
        print(utr5_len, orf_len, utr3_len, pos, length_preset)
        


In [ ]:
m6a_df = m6a_df[m6a_df["label_id"].str.startswith("NM")]

In [ ]:
preset = (mean_utr5, mean_orf, mean_utr3)
# preset = (1, 2, 2)
m6a_df["pos"] = m6a_df["label_id"].str.split(":").str[1].astype(np.int32)
m6a_df["metagene"] = m6a_df.apply(lambda x: calculate_metagene(x["utr5_len"], x["orf_len"], x["utr3_len"], x["pos"], preset), axis=1)

In [ ]:
hela_m6a_df = hela_m6a_df[hela_m6a_df["label_id"].str.startswith("NM")]
preset = (mean_utr5, mean_orf, mean_utr3)
# preset = (1, 2, 2)
hela_m6a_df["pos"] = hela_m6a_df["label_id"].str.split(":").str[1].astype(np.int32)
hela_m6a_df["metagene"] = hela_m6a_df.apply(lambda x: calculate_metagene(x["utr5_len"], x["orf_len"], x["utr3_len"], x["pos"], preset), axis=1)

In [ ]:
## Draw metagene plot
fig, ax = plt.subplots(figsize=(20, 15))

plt.style.use('default')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 22, 'legend.facecolor': 'white', 'legend.framealpha': 1, "legend.frameon": 1, "lines.linewidth": 5})
## Histogram style without fill.

low_cut = 0.25
high_cut = 0.75

low_cut = m6a_df["dom"].quantile(low_cut)
high_cut = m6a_df["dom"].quantile(high_cut)

print(low_cut, high_cut)

sns.kdeplot(m6a_df[m6a_df["drach"]]["metagene"], ax=ax, fill=False, color="royalblue", label="DRACH", zorder=1, bw_adjust=0.75)
sns.kdeplot(m6a_df[~m6a_df["drach"]]["metagene"], ax=ax, fill=False, color="tomato", label="non-DRACH", zorder=3, bw_adjust=0.75)

for i in range(1, 3):
    ax.axvline(np.array(preset[:i]).sum(), color="grey", linestyle="--", zorder=0)
ax.grid(False)
ax.set_xlim(0, 1)
plt.savefig("/extdata4/baeklab/Hyeonseo/m6A/poster/metagene.pdf", format="pdf")
plt.legend()
plt.show()